# Laboratorio 3A: del chat a la API con salidas estructuradas

*Agentes de IA (aplicado) · Sesión 3 · Bloque 3.1 · Especialización en Inteligencia Artificial, UNAL Manizales*

En el Laboratorio 2B, el notebook buscaba «Categoría:» y «Prioridad:» en el texto de cada respuesta. Otro formato bastaba para que la prueba fallara. Los sistemas de tickets de soporte guardan cada caso en campos fijos, como el tipo, la prioridad y la plataforma. En este laboratorio conviertes los tickets de FotoNube en registros que cumplen un esquema, y pruebas cada paso.

**Conceptos clave:**
- **Salida estructurada:** una respuesta del modelo que cumple un esquema, en lugar de texto libre.
- **JSON:** un formato de texto para datos, con claves y valores entre llaves.
- **Esquema:** los campos del registro, su tipo, si son obligatorios y sus valores permitidos.
- **Pydantic:** la biblioteca de Python que escribe el esquema como una clase y valida cada registro.
- **Salida válida:** una respuesta que cumple el esquema.
- **Campo correcto:** un campo cuyo valor coincide con el del registro esperado.

<details>
<summary><b>Repaso: lo que viene del 2A, del 2B y del bloque 3.1</b> (clic para abrir)</summary>

- **LangChain** es la biblioteca de Python con la que el curso llama a los modelos. `init_chat_model` crea un modelo con su nombre y su proveedor.
- **Una llamada** lleva un mensaje de sistema, con las instrucciones fijas, y un mensaje de usuario, con el caso.
- **`usage_metadata`** trae los tokens de entrada y de salida. Con ellos se calcula el costo.
- **Un criterio determinista** da siempre el mismo resultado con la misma respuesta.
- **La tabla de resultados** tiene las mismas columnas en todo el curso. La sección 5 las explica.
- **FotoNube** es la app de fotos ficticia del curso. Sus categorías y sus prioridades son las del 2B.
- **El criterio de éxito va primero:** el bloque 3.1 lo pide antes de escribir el prompt.
- **La decodificación restringida** deja al modelo escribir solo los tokens que cumplen el esquema.
- **La capa 2 del proyecto** es la de las instrucciones: el prompt de sistema y el esquema de tu agente.
- **La capa de evaluación** son las pruebas de cada etapa, de funcionamiento y de robustez. No es una capa del proyecto: acompaña a todas.
</details>

**En este laboratorio vas a:**
- pasar el mismo ticket de texto libre a un registro JSON, con `with_structured_output`;
- leer el esquema `Ticket` como una clase de Pydantic y decidir qué campos son obligatorios;
- probar tres etapas: la respuesta cruda, la validación del esquema y el contenido de los campos;
- provocar tres fallos a propósito: un texto vacío, una respuesta cortada y un ticket sin un dato;
- medir el porcentaje de salidas válidas y el de campos correctos, los tokens y el costo;
- en casa, agregar dos campos y dos casos límite, y correr las pruebas otra vez.

**Resultados de aprendizaje:**
- **RA2:** diseñar prompts y salidas estructuradas, y usar la API de un LLM;
- **RA7:** probar el funcionamiento y la robustez de cada etapa, y estimar su desempeño y su costo.

**Tiempo y entrega.** Son 60 minutos en clase y unas 1.5 horas en casa. En clase haces las secciones 1 a 5. En casa haces las secciones 6 a 10, unos 70 minutos. Los otros 20 son para terminar lo que no alcances en clase. Entregas este notebook ejecutado y guardado con sus salidas, al inicio de la semana siguiente.

## Cómo usar este notebook

- **Ábrelo en VS Code.** Abre la carpeta del laboratorio con «File > Open Folder» y luego abre este archivo.
- **Elige el kernel.** Arriba a la derecha, en «Select Kernel», elige el entorno virtual del curso.
- **Ejecuta en orden,** con Shift+Enter. Cada celda usa lo que hicieron las anteriores. Si reinicias el kernel, vuelve a ejecutar desde el principio.
- **Edita solo las celdas con ✏️:** los ejercicios, la configuración y tus análisis.
- **Si algo falla, lee el mensaje:** dice qué revisar. Si el error sigue, reinicia el kernel y ejecuta todo de nuevo. También puedes usar el modo grabado de la sección 1.
- **Guarda con las salidas,** con Ctrl+S (Cmd+S en Mac). El docente califica sin volver a ejecutar.

<details>
<summary><b>¿No puedes instalar VS Code? Usa Google Colab como respaldo</b> (clic para abrir)</summary>

Colab sirve solo en modo grabado.
1. En Colab, abre este notebook con «Archivo > Subir notebook».
2. En el panel «Archivos», sube el archivo `lab_3A_salidas_estructuradas.zip`: el mismo .zip del laboratorio que descargaste.
3. Ejecuta la primera celda de la sección 1: descomprime el archivo y entra a su carpeta.

El archivo `LEEME.md` de la carpeta explica los pasos con más detalle.
</details>

## Contenido

* [Datos de la entrega](#0)
* [1 - Preparar el entorno](#1) · en clase, 5 min
* [2 - Del chat a la API](#2) · en clase, 12 min
    * [2.1 - El ticket guía, en texto libre](#2-1)
    * [2.2 - El JSON y el esquema](#2-2)
    * [2.3 - La primera lectura con with_structured_output](#2-3)
* [3 - El esquema y la instrucción](#3) · en clase, 13 min
    * [3.1 - Los doce tickets](#3-1)
    * [3.2 - Las dos instrucciones](#3-2)
    * [3.3 - Ejercicio 1: tu configuración](#3-3)
* [4 - Las pruebas](#4) · en clase, 22 min
    * [4.1 - Ejercicio 2: dos registros esperados](#4-1)
    * [4.2 - La respuesta cruda](#4-2)
    * [4.3 - La validación del esquema](#4-3)
    * [4.4 - El contenido de los campos](#4-4)
* [5 - Los resultados](#5) · en clase, 8 min
* [6 - Ejercicio 3: revisa al revisor](#6) · para casa, 10 min
* [7 - Para casa: dos campos y dos casos límite](#7) · para casa, 40 min
    * [7.1 - Ejercicio 4: dos campos nuevos](#7-1)
    * [7.2 - Ejercicio 4: dos casos límite](#7-2)
    * [7.3 - Las pruebas otra vez](#7-3)
* [8 - Entregable y calificación](#8) · para casa, 5 min
* [9 - Comprobación de la entrega](#9) · para casa, 5 min
* [10 - Tu proyecto: la capa 2](#10) · para casa, 10 min
* [Conclusión](#conclusion)
* [Referencias](#referencias)

<details>
<summary><b>💡 Otra forma de navegar: el panel Esquema de VS Code</b> (clic para abrir)</summary>

VS Code muestra los títulos del notebook en el panel «Esquema» («Outline» en inglés). Lo encuentras en la barra lateral del Explorador, abajo. Haz clic en un título para saltar a esa sección. Si un enlace de esta tabla no funciona en tu versión de VS Code, usa el panel.
</details>

<a id='0'></a>
## Datos de la entrega

Escribe tu nombre, o los de la pareja si el docente lo permite. Estos datos se quedan en el notebook: nunca se envían a un modelo. El modo, el modelo y la fecha se agregan solos.

* ✏️ Escribe tu nombre en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Datos de la entrega: escribe tu nombre entre las comillas
datos_entrega = {
    "nombres": "...",           # por ejemplo, "Ana Pérez" o "Ana Pérez y Luis Gómez"
    "modalidad": "individual",  # "individual" o "pareja", según defina el docente
}

<a id='1'></a>
## 1 - Preparar el entorno

*En clase · 5 minutos*

El arranque es el mismo de los laboratorios 2A y 2B. La celda siguiente revisa la versión de Python, instala los paquetes que falten y busca los archivos del laboratorio. En Colab, antes descomprime el archivo .zip.

* Ejecuta la siguiente celda para comprobar el entorno.

In [ ]:
# Comprueba la versión de Python, los paquetes del curso y los archivos del laboratorio
import importlib.util
import subprocess
import sys
from pathlib import Path

# En Colab: si subiste el .zip del laboratorio, lo descomprime y entra a su carpeta
if "google.colab" in sys.modules and not Path("datos").exists():
    import os, zipfile
    for archivo_zip in Path("/content").glob("lab_3A*.zip"):
        zipfile.ZipFile(archivo_zip).extractall("/content")
        os.chdir("/content/lab_3A_salidas_estructuradas")

print(f"Python {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("❌ Este laboratorio necesita Python 3.11 o posterior. Elige el kernel del entorno del curso.")

# Instala en el kernel solo los paquetes que faltan, con las versiones del curso
paquetes = {"langchain": "langchain", "langchain_anthropic": "langchain-anthropic",
            "langchain_google_genai": "langchain-google-genai", "pydantic": "pydantic", "pandas": "pandas",
            "matplotlib": "matplotlib", "dotenv": "python-dotenv"}
versiones_curso = {}
if Path("requirements.txt").exists():
    for linea in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
        if "==" in linea:
            versiones_curso[linea.split("==")[0].strip()] = linea.strip()
faltan = [versiones_curso.get(pip, pip) for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalo lo que falta:", ", ".join(faltan))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=False)
    print("Listo. Reinicia el kernel y vuelve a ejecutar desde el principio.")
else:
    print("✅ Los paquetes del curso están instalados.")

# Revisa los archivos del laboratorio
for ruta in ["curso_agentes", "datos/tickets_3A.json", "grabaciones"]:
    print(("✅ " if Path(ruta).exists() else "❌ falta ") + ruta)

Si ves una ❌, sigue su indicación. Si falta un archivo, abre la carpeta completa del laboratorio, no solo el notebook.

**El modo y el modelo.** Como en el 2A, el notebook corre en modo grabado, sin clave, o en vivo, con la clave del curso. El modelo por defecto es Claude Haiku 5.5. Si tu grupo eligió Gemini en la capa 1, usa Gemini 3.5 Flash-Lite: también tiene grabaciones.

> **Nota:** la clave vive en el archivo `.env`. Nunca la escribas en una celda: el notebook se entrega con todo lo que contiene.

* ✏️ Si quieres, cambia el modo, el modelo o los límites. Luego ejecuta la siguiente celda.

In [ ]:
# ✏️ Elige el modo y el modelo. Con "auto", el notebook usa el modo en vivo solo si tu clave sirve
MODO = "auto"                           # "auto", "grabado" o "en_vivo"
MODELO = "anthropic:claude-haiku-5-5"   # o "google_genai:gemini-3.5-flash-lite"
LIMITE_LLAMADAS = 120                   # en vivo, el máximo de llamadas de esta corrida
LIMITE_TOKENS = 300_000                 # en vivo, el máximo de tokens de esta corrida

import pandas as pd
import curso_agentes as ca

config = ca.configurar(modo=MODO, modelo=MODELO, max_llamadas=LIMITE_LLAMADAS, max_tokens=LIMITE_TOKENS)
ca.mostrar_ficha_entrega(datos_entrega, config)

En vivo, `init_chat_model` crea el modelo con su nombre, su proveedor y sus parámetros. En modo grabado, el módulo del curso crea un modelo simulado de LangChain que se usa igual.

* Ejecuta la siguiente celda para crear el modelo.

In [ ]:
# Crea el modelo: en vivo con init_chat_model, o el modelo simulado con sus grabaciones
from langchain.chat_models import init_chat_model

if config.modo == "en_vivo":
    modelo = init_chat_model(config.modelo, **config.parametros)
else:
    modelo = ca.modelo_grabado(config)

print("Modelo listo:", config.nombre_modelo, "·", type(modelo).__name__)
print("Parámetros:", config.parametros)
print("Versiones:", ca.versiones())

Los parámetros y las versiones quedan en las salidas, junto a tus resultados.

<details>
<summary><b>Para profundizar: los parámetros y el modo grabado</b> (clic para abrir)</summary>

Con Claude Haiku 5.5, el curso desactiva el razonamiento (`thinking`) y fija 512 tokens de salida. El modelo no acepta `temperature`, así que no se envía.

El modelo simulado es una subclase de `BaseChatModel`, como propone la guía de pruebas de LangChain. Desde esta sesión también acepta `with_structured_output`, con los mismos argumentos que un modelo en vivo. La grabación guarda la respuesta cruda del modelo real. Al reproducirla, el módulo la vuelve a leer con el mismo analizador de LangChain, que la valida con Pydantic. La huella de cada grabación incluye el esquema. Si cambias un campo, ese caso queda sin grabación, y el notebook lo dice.
</details>

<a id='2'></a>
## 2 - Del chat a la API

*En clase · 12 minutos*

En una app de chat, la app agrega instrucciones que no ves. Por API, tu programa arma todos los mensajes y recibe la respuesta.

<a id='2-1'></a>
### 2.1 - El ticket guía, en texto libre

El ticket guía de la sesión es este:

> Desde ayer la app se cierra cada vez que subo una foto desde la galería. Uso un Pixel 8 con Android 15 y la versión 4.2.1. Con fotos pequeñas no pasa.

Primero lo envías sin esquema, con una instrucción corta. La respuesta llega como texto libre.

* Ejecuta la siguiente celda para cargar los tickets y enviar el ticket guía en texto libre.

In [ ]:
# Carga los datos del laboratorio y envía el ticket guía sin esquema, como texto libre
datos = ca.cargar_tickets()
guia = next(c for c in datos["base"] if c["caso_id"] == datos["caso_guia"])

null, true, false = None, True, False   # así se escriben en JSON; en Python son None, True y False. Con esta línea sirven las dos formas

mensajes = [("system", datos["instruccion_texto_libre"]), ("user", guia["entrada"])]
respuesta_libre = modelo.invoke(mensajes)
print("Texto:", ca.texto_de(respuesta_libre))      # el texto de respuesta_libre.content
print("Tokens:", respuesta_libre.usage_metadata)

La respuesta sirve a una persona. Un programa, en cambio, no sabe dónde está la prioridad, ni si la plataforma es Android. En el 2B, el notebook tuvo que buscar palabras en el texto. Ahora pides un registro.

<a id='2-2'></a>
### 2.2 - El JSON y el esquema

Un **registro** guarda cada dato en un campo con nombre. En **JSON**, cada campo tiene una clave, dos puntos y un valor, y `null` quiere decir «vacío».

```json
{"categoria": "fallo de la app", "prioridad": "alta", "plataforma": "Android", "version_app": "4.2.1",
 "resumen": "La app se cierra al subir fotos grandes desde la galería"}
```

El **esquema** dice qué campos lleva el registro, de qué tipo son y qué valores aceptan. Es el de las diapositivas:

| Campo | Tipo | ¿Obligatorio? | Valores permitidos |
| :--- | :--- | :--- | :--- |
| categoria | texto | sí | fallo de la app, rendimiento, interfaz, cuenta y acceso, solicitud de función |
| prioridad | texto | sí | alta, media, baja |
| plataforma | texto | sí | Android, iOS, web |
| version_app | texto o vacío | no | null si el ticket no la dice |
| resumen | texto | sí | una frase |

En Python, el esquema se escribe como una clase de **Pydantic**. No tienes que escribirla: basta con leerla.
- `Literal[...]` quiere decir «uno de estos valores».
- `str` es texto, y `None` es vacío.
- `str | None = None` quiere decir «texto, o vacío si no viene».

* Ejecuta la siguiente celda para definir el esquema `Ticket`.

In [ ]:
# Define el esquema Ticket con Pydantic, igual que en las diapositivas
from typing import Literal
from pydantic import BaseModel

class Ticket(BaseModel):
    categoria: Literal["fallo de la app", "rendimiento", "interfaz", "cuenta y acceso", "solicitud de función"]
    prioridad: Literal["alta", "media", "baja"]
    plataforma: Literal["Android", "iOS", "web"]
    version_app: str | None = None
    resumen: str

print("Campos del esquema:", list(Ticket.model_fields))

**Obligatorio quiere decir que el campo siempre trae un valor.** Si el ticket no trae el dato, el modelo lo inventa o escribe un relleno, como «desconocida». En este esquema, `version_app` puede quedar vacía, pero la `plataforma` no.

<a id='2-3'></a>
### 2.3 - La primera lectura con `with_structured_output`

LangChain pide un registro en una línea:

```python
lector = modelo.with_structured_output(Ticket, method="json_schema", include_raw=True)
```

- `Ticket` es el esquema. LangChain lo envía junto con los mensajes.
- `method="json_schema"` usa las salidas estructuradas del proveedor: el modelo solo puede escribir tokens que cumplen el esquema.
- `include_raw=True` devuelve tres cosas: `"raw"`, la respuesta cruda; `"parsed"`, el registro, o `None` si falló; y `"parsing_error"`, el error al leerla.

<details>
<summary><b>Para profundizar: por qué json_schema</b> (clic para abrir)</summary>

Con Claude, LangChain usa por defecto una herramienta forzada. Si el modelo razona, LangChain ya no la fuerza. Con `method="json_schema"`, en cambio, usa las salidas estructuradas de Claude: el esquema viaja en `output_config`. Con Gemini, `json_schema` ya es el método por defecto.
</details>

* Ejecuta la siguiente celda para pedir el registro del ticket guía y medir la llamada.

In [ ]:
# Pide el registro del ticket guía con el esquema Ticket, y mide la latencia
import time

lector = modelo.with_structured_output(Ticket, method="json_schema", include_raw=True)
mensajes = [("system", datos["instrucciones"]["basica"]), ("user", guia["entrada"])]

inicio = time.perf_counter()
respuesta = lector.invoke(mensajes)
latencia = time.perf_counter() - inicio

ca.mostrar_lectura(respuesta, config, latencia, esquema=Ticket)

A la izquierda está lo que escribió el modelo. A la derecha, el registro que armó Pydantic, campo por campo. Con el registro, un programa lee cada dato sin buscar palabras. El punto (·) junto a cada campo quiere decir que cumple el esquema. Si el valor es el correcto lo prueba la sección 4.4.

* Ejecuta la siguiente celda para leer dos campos del registro y los tokens de la respuesta cruda.

In [ ]:
# Lee dos campos del registro, como lo haría un programa, y los tokens de la respuesta cruda
registro = respuesta["parsed"]
if registro is not None:
    print("Prioridad:", registro.prioridad, "· Plataforma:", registro.plataforma)
else:
    print("El registro no llegó:", respuesta["parsing_error"])
print("Tokens:", respuesta["raw"].usage_metadata)

**¿Siempre llega un registro válido?** Casi siempre, no siempre. Los tres proveedores grandes garantizan la forma, con excepciones, pero ninguno garantiza el contenido. Por eso el programa **valida otra vez** cada registro con Pydantic, y luego compara cada campo con el esperado.

<details>
<summary><b>Qué garantiza cada proveedor, según el bloque 3.1</b> (clic para abrir)</summary>

| Proveedor | ¿Garantiza el esquema? | Lo que no garantiza |
| :--- | :--- | :--- |
| Claude | Sí, con decodificación restringida, «in most cases» | Un rechazo, un corte por `max_tokens` o las mayúsculas de un valor permitido |
| Gemini | Sí: un JSON válido que cumple el esquema | Que los valores sean correctos. Además, ignora las reglas del esquema que no admite |
| OpenAI | Sí, con `strict: true`, desde agosto de 2024 | Un rechazo, un corte por `max_tokens` y los errores dentro de los valores |

*Fuentes: [salidas estructuradas de Claude](https://platform.claude.com/docs/en/build-with-claude/structured-outputs), [salidas estructuradas de Gemini](https://aistudio.google.com/docs/structured-output?generate-content=true) y [OpenAI (2024)](https://openai.com/index/introducing-structured-outputs-in-the-api/). Consulta: 24 de noviembre de 2026.*
</details>

<a id='3'></a>
## 3 - El esquema y la instrucción

*En clase · 13 minutos*

<a id='3-1'></a>
### 3.1 - Los doce tickets

**FotoNube** es la app de fotos ficticia de los laboratorios 2A y 2B. Las prioridades siguen las reglas del 2B:

| Prioridad | Cuándo |
| :--- | :--- |
| alta | La app se cierra o se congela, aunque sea una sola vez; se pierden fotos o cambios; o falla el acceso a la cuenta |
| media | Una función falla o la app va lenta, pero la persona puede seguir usándola |
| baja | Detalles visuales, textos o sugerencias |

Si un ticket trae varios problemas, se clasifica el más grave. La plataforma es Android para los equipos Android, iOS para iPhone y iPad, y web para el navegador. **Un dato que el ticket no trae queda vacío (null).** Ojo: el esquema de las diapositivas hace obligatoria la plataforma. Con ese esquema, T-06 y T-10 no pueden cumplir esta regla. Lo pruebas en la sección 4.

<details>
<summary><b>Las cinco categorías</b> (clic para abrir)</summary>

| Categoría | Qué significa |
| :--- | :--- |
| fallo de la app | La app se cierra, se congela o una función no hace lo que debe |
| rendimiento | La app funciona, pero va lenta o gasta mucha batería, memoria o datos |
| interfaz | Textos, colores, botones o pantallas que se ven mal |
| cuenta y acceso | Problemas para entrar, registrarse o recuperar la cuenta |
| solicitud de función | Una idea o un pedido de algo que la app no tiene |
</details>

* Ejecuta la siguiente celda para ver los doce tickets, con su registro esperado.

In [ ]:
# Muestra los doce tickets, con su tipo de caso y su registro esperado
ca.mostrar_tickets(datos["base"])

Cinco tickets son **normales**. Los otros siete son difíciles a propósito: de borde, en otro idioma, mal escritos, incompletos o ambiguos. Cada etapa tiene sus propias pruebas de robustez. Por eso, un mismo ticket puede ser de funcionamiento en una etapa y de robustez en otra. El registro de T-11 y T-12 lo escribes tú en el Ejercicio 2. Todos los tickets son ficticios y no traen datos personales.

<a id='3-2'></a>
### 3.2 - Las dos instrucciones

La instrucción es el mensaje de sistema. El laboratorio trae dos:
- **básica:** una instrucción clara, de tres oraciones;
- **con reglas:** la misma, más el contexto. Trae las categorías, las prioridades y la plataforma, la regla de no inventar y el largo del resumen.

**Desglose del experimento**

| | Instrucción básica | Instrucción con reglas |
| :--- | :--- | :--- |
| **Qué trae** | La tarea y el esquema | Lo mismo, más las reglas y «si un dato no está, déjalo vacío» |
| **Qué debería pasar** | Registros válidos, con más errores de prioridad | Más campos correctos |
| **Cómo puede fallar** | El modelo adivina un dato que el ticket no trae | Aplica mal una regla en un ticket de borde. Y si el esquema hace obligatorio un campo, no puede dejarlo vacío |

* Ejecuta la siguiente celda para leer las dos instrucciones completas.

In [ ]:
# Muestra las dos instrucciones, tal como llegan al modelo
for nombre, texto in datos["instrucciones"].items():
    print(f"===== Instrucción {ca.INSTRUCCIONES_3A[nombre]} =====\n{texto}\n")

<a id='3-3'></a>
### 3.3 - Ejercicio 1: tu configuración

**Qué se pide.** Elige la instrucción y decide si dos campos del esquema son obligatorios:
- `PLATAFORMA_OBLIGATORIA`: `True` si la plataforma siempre debe traer un valor, o `False` si puede quedar vacía;
- `VERSION_OBLIGATORIA`: lo mismo para `version_app`.

**Por qué.** El esquema y la instrucción trabajan juntos. Si el esquema hace obligatorio un campo, el modelo pone un valor aunque la instrucción pida dejarlo vacío.

**Cómo saber si quedó bien.** No hay una elección correcta: es un experimento. La celda de revisión muestra tu esquema. Antes de correr, escribe tu predicción.

> **Nota:** en casa, todos parten de la instrucción con reglas, con la plataforma y la versión que pueden quedar vacías. Si en clase eliges otra configuración, tendrás dos puntos de comparación.

* ✏️ Completa la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 1: elige la instrucción y decide qué campos son obligatorios
INSTRUCCION = ...              # "basica" o "con_reglas"
PLATAFORMA_OBLIGATORIA = ...   # True: siempre trae un valor. False: puede quedar vacía (None)
VERSION_OBLIGATORIA = ...      # True o False, sin comillas

**✏️ Tu predicción.** En una o dos oraciones: ¿qué pondrá el modelo en la plataforma de T-06, que no la dice? ¿Por qué?

*Escribe aquí tu predicción.*

La celda siguiente revisa tu elección, arma tu esquema y crea el lector.

* Ejecuta la siguiente celda para armar tu esquema y tu lector.

In [ ]:
# Revisa tu configuración, arma tu esquema y crea el lector con with_structured_output
instruccion_usada, plataforma_obligatoria, version_obligatoria = ca.revisar_configuracion(
    INSTRUCCION, PLATAFORMA_OBLIGATORIA, VERSION_OBLIGATORIA)
Esquema = ca.armar_ticket(plataforma_obligatoria, version_obligatoria)
sistema = datos["instrucciones"][instruccion_usada]
variante = ca.variante_de(instruccion_usada, Esquema)
ca.mostrar_esquema(Esquema, titulo="Tu esquema")
lector = modelo.with_structured_output(Esquema, method="json_schema", include_raw=True)

<a id='4'></a>
## 4 - Las pruebas

*En clase · 22 minutos*

La capa de evaluación prueba cada etapa por separado, en orden. Si falla una etapa temprana, las siguientes no tienen qué revisar.

| Etapa | Prueba de funcionamiento | Prueba de robustez | Criterio de aceptación |
| :--- | :--- | :--- | :--- |
| Respuesta cruda | El modelo devuelve una salida para cada texto | Texto vacío o en otro idioma | Hay una salida, o un error registrado |
| Validación del esquema | La salida cumple el esquema | Una respuesta mal formada | El esquema valida. Si no valida, el error se detecta y se reintenta |
| Contenido de los campos | Cada campo trae el valor esperado | Texto sin un campo obligatorio | Cada campo coincide con el esperado. Un campo ausente queda vacío, no inventado |

«Texto sin un campo obligatorio» es un ticket que no trae un dato que el esquema pide, como la plataforma. Si ese campo puede quedar vacío, debe quedar en null. Si es obligatorio, el modelo tendrá que inventar.

<a id='4-1'></a>
### 4.1 - Ejercicio 2: dos registros esperados

**Qué se pide.** Escribe el registro esperado de T-11 y T-12: la categoría, la prioridad, la plataforma y la versión. Usa las reglas de la sección 3.1. Si el ticket no trae un dato, escribe `None`, sin comillas. En JSON el vacío es `null`, y en Python es `None`: el notebook acepta los dos.

**Por qué.** Sin registro esperado no hay prueba de contenido.

**Cómo saber si quedó bien.** La celda que arma la lista dice «2 de 2 registros esperados del Ejercicio 2», sin avisos ni pistas.

<details>
<summary><b>Pista</b> (clic para abrir)</summary>

- T-11 trae dos problemas. Clasifica el más grave, y usa la plataforma de ese problema.
- T-12 pide algo que la app no tiene. Revisa qué sistema usa el iPhone.
- La versión es el número que trae el ticket, aunque le falte una parte.
</details>

* ✏️ Completa los dos registros de la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 2: el registro esperado de T-11 y T-12. Si el ticket no trae un dato, escribe None
mis_registros = [
    {"caso_id": "T-11", "esperado": {"categoria": ..., "prioridad": ..., "plataforma": ..., "version_app": ...}},
    {"caso_id": "T-12", "esperado": {"categoria": ..., "prioridad": ..., "plataforma": ..., "version_app": ...}},
]

La celda siguiente une los doce tickets con tus dos registros. Si un valor no sigue las reglas, te da una pista sin decirte la respuesta.

* Ejecuta la siguiente celda para armar la lista de casos.

In [ ]:
# Une los doce tickets con tus dos registros esperados y revisa cada campo
casos = ca.armar_casos_3A(datos, mis_registros, modo=config.modo)
ca.mostrar_tickets([c for c in casos if c["origen"] == "tuya"])

<a id='4-2'></a>
### 4.2 - La respuesta cruda

Ahora cada ticket va al modelo una vez, con tu instrucción y tu esquema. En vivo, son 12 llamadas. La primera etapa solo mira si llegó una salida.

**Desglose del experimento**

| | |
| :--- | :--- |
| **Entrada** | Los doce tickets, con tu configuración |
| **Prueba de funcionamiento** | Cada ticket recibe una salida, con texto y tokens |
| **Pruebas de robustez** | T-08, escrito en inglés, y un texto vacío |
| **Qué debería pasar** | T-08 también recibe un registro. El notebook no envía un texto vacío. Aparte, registra qué hace el proveedor si se envía |

* Ejecuta la siguiente celda para enviar los doce tickets.

In [ ]:
# Envía cada ticket una vez con tu lector, y guarda cada lectura
lecturas = ca.correr_tickets(lector, config, casos, sistema, Esquema, variante=variante, modelo=modelo)

**El texto vacío** tiene dos pruebas, como en el 2A. `R-VACIO` comprueba que el notebook no lo envía. `R-VACIO-API` lo envía a propósito y registra qué hace el proveedor.

* Ejecuta la siguiente celda para correr las pruebas del texto vacío y ver la etapa de la respuesta cruda.

In [ ]:
# Corre las dos pruebas del texto vacío y muestra la etapa de la respuesta cruda
vacio = ca.prueba_texto_vacio(lector, config, sistema, Esquema, variante=variante)
ca.mostrar_etapa(ca.tabla_resultados_3A(lecturas, casos, None, vacio=vacio), "respuesta_cruda")

<br>

### Análisis

La etapa pasa si cada ticket recibió una salida, también T-08, que llegó en inglés. Si `R-VACIO-API` muestra un error del proveedor, la prueba pasa. El error quedó registrado, y nadie lo confunde con una respuesta.

<a id='4-3'></a>
### 4.3 - La validación del esquema

La segunda etapa mira si cada salida cumple el esquema. El módulo la valida otra vez con Pydantic, y explica cada error en español. Las mayúsculas cuentan: «Solicitud de función», con mayúscula, no es un valor permitido, y la salida queda inválida.

* Ejecuta la siguiente celda para ver la etapa del esquema.

In [ ]:
# Muestra la etapa del esquema y, si hubo, la primera salida inválida en detalle
ca.mostrar_etapa(ca.tabla_resultados_3A(lecturas, casos, None, vacio=vacio), "esquema",
                 nota="La prueba de robustez de esta etapa, la respuesta cortada, viene en la celda siguiente.")
invalidas = [l for l in lecturas if not l.valida and not l.error]
if invalidas:
    ca.mostrar_lectura(invalidas[0], config, esquema=Esquema, titulo=f"La primera salida inválida: {invalidas[0].caso_id}")

**La respuesta mal formada.** Casi todas las salidas son válidas, así que esta prueba provoca el fallo a propósito. Un máximo de 24 tokens de salida corta el JSON antes de terminar. El módulo detecta el error y reintenta una vez con el máximo normal.

> **Nota:** con el razonamiento activo, un máximo pequeño puede agotarse antes de escribir cualquier texto. Por eso el curso lo desactiva.

* Ejecuta la siguiente celda para provocar la respuesta cortada y su reintento.

In [ ]:
# Provoca una respuesta cortada con un máximo de tokens muy bajo, y reintenta con el máximo normal
if config.modo == "en_vivo":
    modelo_corto = init_chat_model(config.modelo, **{**config.parametros, "max_tokens": datos["max_tokens_corto"]})
else:
    modelo_corto = ca.modelo_grabado(config, escenario="max_tokens_bajo")
lector_corto = modelo_corto.with_structured_output(Esquema, method="json_schema", include_raw=True)

cortada = ca.prueba_respuesta_cortada(lector_corto, lector, config, sistema, Esquema, guia, variante=variante)
ca.mostrar_lectura(cortada[0], config, esquema=Esquema, titulo="Intento 1: la respuesta cortada")

<br>

### Análisis

`stop_reason` dice `max_tokens` en Claude y `MAX_TOKENS` en Gemini: la respuesta se detuvo por el límite. La prueba pasa si la validación detecta el corte y el reintento trae un registro válido. Sin esa validación, un programa guardaría un registro roto.

<a id='4-4'></a>
### 4.4 - El contenido de los campos

La última etapa compara cada campo con el registro esperado, siempre de la misma forma:
- **categoría, prioridad y plataforma:** deben coincidir con el esperado. Sus mayúsculas ya las revisó el esquema;
- **versión:** debe coincidir el número, así que «v4.2.1» cuenta como «4.2.1»;
- **resumen:** basta que exista y que no pase de 20 palabras. La instrucción pide 15, y el criterio deja un margen. La sección 6 revisa ese límite;
- **un dato que el ticket no trae:** debe quedar vacío (null). Un relleno, como «desconocida», no cuenta.

> **Nota:** los campos se comparan solo en los registros válidos. Una salida inválida ya falló en la etapa del esquema.

* Ejecuta la siguiente celda para comparar cada campo con el esperado.

In [ ]:
# Compara cada campo de cada registro válido con su registro esperado
detalle = ca.comparar_campos(lecturas, casos, Esquema)
ca.mostrar_campos(detalle)

**La prueba de robustez de esta etapa** son los campos que el ticket no trae. T-06 y T-10 no dicen la plataforma, y varios tickets no dicen la versión.

* Ejecuta la siguiente celda para ver qué hizo el modelo con cada dato que el ticket no trae.

In [ ]:
# Muestra los campos sin dato en el ticket: ¿quedaron vacíos, o el modelo puso un valor?
ca.mostrar_ausentes(detalle)

<br>

### Análisis

- **Si la plataforma falla en T-06 o T-10 y es obligatoria,** el esquema obligó al modelo a poner un valor.
- **Si un campo que puede quedar vacío trae un valor inventado,** la instrucción no bastó.
- **Si fallan la categoría o la prioridad en un ticket claro,** el fallo es del modelo.

**✏️ Tu análisis (4).** En tres a cinco oraciones, con casos de la tabla: ¿se cumplió tu predicción del Ejercicio 1? ¿Qué campo falló más? ¿El fallo vino del esquema, de la instrucción o del modelo?

*Escribe aquí tu análisis.*

<a id='5'></a>
## 5 - Los resultados

*En clase · 8 minutos*

La tabla de resultados junta las tres etapas, con las columnas que usa todo el curso. Viene plegada: ábrela para ver cada fila.

<details>
<summary><b>Las columnas de la tabla de resultados</b> (clic para abrir)</summary>

Son las mismas en todo el curso:
- `sesion`, `laboratorio` y `variante`: dónde se corrió la prueba y con qué configuración;
- `forma`: `llm_directo` en toda la sesión 3. Las sesiones 4 a 7 agregan `flujo` y `agente`;
- `caso_id`, `etapa` y `tipo_prueba`: qué caso, qué etapa (`respuesta_cruda`, `esquema` o `campos`) y qué tipo de prueba;
- `esperado`, `obtenido`, `resultado` y `razon_fallo`: qué se esperaba, qué salió, si pasa y por qué falla;
- `intento`, `modo` y `modelo`: el número de intento, grabado o en vivo, y el modelo;
- `tokens_entrada`, `tokens_salida`, `costo_usd` y `latencia_s`: el consumo. Su valor va solo en la fila de la respuesta cruda, porque las otras dos etapas revisan la misma llamada;
- `pasos`, `llamadas_herramientas` e `intervenciones_humanas`: el comportamiento de un agente. Aquí valen 1, 0 y 0;
- `fecha`: cuándo se corrió. `tipo_caso` y `campos_correctos` son columnas propias de este laboratorio.
</details>

Las medidas resumen las pruebas:
- **salidas válidas:** las salidas que cumplen el esquema, de todos los tickets;
- **campos correctos:** los campos que coinciden con el esperado, de los registros válidos;
- **tokens, costo y latencia:** el consumo de las doce llamadas de los tickets.

* Ejecuta la siguiente celda para ver la tabla de resultados, las medidas y su gráfico.

In [ ]:
# Arma la tabla de resultados con las tres etapas y muestra las medidas del laboratorio
resultados = ca.tabla_resultados_3A(lecturas, casos, detalle, vacio=vacio, cortada=cortada)
ca.mostrar_resultados_3A(resultados, plegada=True)
resumen = ca.resumen_3A(lecturas, detalle)
ca.mostrar_resumen_3A(resumen)
ca.grafico_campos(lecturas, detalle, titulo=f"Tu configuración: {variante}")

El costo de la corrida suma también las dos llamadas de la respuesta cortada. No incluye las dos llamadas de prueba de la sección 2.

* Ejecuta la siguiente celda para ver el costo de tu corrida.

In [ ]:
# Suma las llamadas, los tokens y el costo de toda la corrida
costo = ca.costo_de_la_corrida(resultados)

<br>

### Análisis

Las dos medidas cuentan cosas distintas. Un 100 % de salidas válidas dice que el programa puede leer cada registro. No dice que los registros estén bien: eso lo dicen los campos correctos. Con salidas válidas altas y campos correctos bajos, el problema está en el contenido, no en la forma.

<a id='6'></a>
## 6 - Ejercicio 3: revisa al revisor

*Para casa · 10 minutos*

Un criterio automático también puede equivocarse. El del resumen solo mira que exista y que sea corto. No sabe si dice lo mismo que el ticket, ni si está en español.

**Qué se pide.** Revisa tres resúmenes y di si estás de acuerdo con el criterio automático. Incluye T-08, el ticket en inglés. Si los tres están bien, en el tercer motivo escribe un resumen que pasaría el criterio y estaría mal.

**Por qué.** Antes de confiar en un criterio, se revisa a mano una muestra, como en el Ejercicio 3 del 2A.

**Cómo saber si quedó bien.** La tabla final muestra tus tres revisiones, cada una con su motivo.

* Ejecuta la siguiente celda para ver cada ticket junto a su resumen.

In [ ]:
# Muestra cada ticket junto al resumen del modelo y al resultado del criterio automático
ca.tabla_para_revisar_3A(lecturas, detalle, casos)

* ✏️ Completa las tres revisiones de la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 3: tres revisiones del resumen. de_acuerdo: True o False, sin comillas
revision = [
    {"caso_id": "T-08", "campo": "resumen", "de_acuerdo": None, "motivo": "..."},
    {"caso_id": ..., "campo": "resumen", "de_acuerdo": None, "motivo": "..."},
    {"caso_id": ..., "campo": "resumen", "de_acuerdo": None, "motivo": "..."},
]

* Ejecuta la siguiente celda para comparar tu revisión con el criterio automático.

In [ ]:
# Compara tu revisión con el criterio automático
revisadas = ca.mostrar_revision_3A(revision, detalle)

Si estás en desacuerdo con el criterio en algún caso, encontraste su límite. En tu proyecto, un campo de texto libre necesita otra prueba. Por ejemplo, revisar a mano una muestra, o pedir que el resumen nombre un dato clave del ticket.

<a id='7'></a>
## 7 - Para casa: dos campos y dos casos límite

*Para casa · unos 40 minutos*

Ahora amplías el esquema, como pide el trabajo independiente del bloque 3.1:
- agregas dos campos al esquema, con su nombre, su tipo y si son obligatorios;
- escribes dos casos límite: textos en el borde de una regla del esquema;
- corres las pruebas otra vez y comparas.

En casa, todos parten de la misma configuración. Es la instrucción con reglas, con la plataforma y la versión que pueden quedar vacías. Así tus campos son el único cambio.
- **Sin clave,** eliges los campos de un banco y los casos de otro. Los dos bancos tienen respuestas grabadas.
- **Con la clave del curso,** también puedes definir un campo propio y escribir tus propios tickets.

<a id='7-1'></a>
### 7.1 - Ejercicio 4: dos campos nuevos

El banco trae cuatro campos, con tipos distintos: una lista de valores permitidos, texto, y verdadero o falso.

* Ejecuta la siguiente celda para ver el banco de campos.

In [ ]:
# Muestra los cuatro campos del banco, con su tipo y lo que contienen
ca.mostrar_banco_campos(datos)

**Qué se pide.** Elige dos campos y decide si cada uno es obligatorio. Piensa en los tickets: ¿todos traen ese dato?

**Por qué.** Cada campo nuevo es una pregunta más para el modelo. Si es obligatorio y el dato falta, el modelo tendrá que poner un valor.

**Cómo saber si quedó bien.** La celda de revisión muestra tu esquema nuevo y la plantilla de tus casos límite.

<details>
<summary><b>Con la clave del curso: un campo propio</b> (clic para abrir)</summary>

En vivo puedes definir un campo que no está en el banco. Escribe su `nombre`, su `tipo` y su `descripcion`. El tipo puede ser `"texto"`, `"numero"`, `"verdadero o falso"` o una lista de valores permitidos. Por ejemplo:

```python
{"nombre": "sentimiento", "tipo": ["positivo", "neutro", "negativo"], "obligatorio": True,
 "descripcion": "El tono del ticket."}
```

Un campo propio no tiene valor esperado en los doce tickets. Por eso se compara solo en tus casos límite.
</details>

* ✏️ Elige tus dos campos en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 4: dos campos nuevos del banco. obligatorio: True o False, sin comillas
# Copia el nombre de la tabla del banco. Por ejemplo: {"nombre": "idioma", "obligatorio": True}
CAMPOS_NUEVOS = [
    {"nombre": ..., "obligatorio": ...},
    {"nombre": ..., "obligatorio": ...},
]

* Ejecuta la siguiente celda para revisar tus campos y armar tu esquema nuevo.

In [ ]:
# Revisa tus dos campos y arma el esquema de casa
casa = datos["casa"]
campos_nuevos = ca.armar_campos_nuevos(CAMPOS_NUEVOS, datos, modo=config.modo)
EsquemaCasa = ca.armar_ticket(casa["plataforma_obligatoria"], casa["version_obligatoria"], campos_nuevos)
ca.mostrar_esquema(EsquemaCasa, titulo="Tu esquema de casa")

<a id='7-2'></a>
### 7.2 - Ejercicio 4: dos casos límite

En tu esquema de casa, `Field(description=...)` agrega la descripción de cada campo nuevo. La descripción viaja dentro del esquema, así que el modelo también la lee.

Un caso límite queda en el borde de una regla. Por ejemplo, un ticket que no dice la plataforma, o una versión con letras.

* Ejecuta la siguiente celda para ver el banco de casos límite y tu plantilla.

In [ ]:
# Muestra los seis casos límite del banco, sin su registro esperado, y la plantilla con tus dos campos
ca.mostrar_banco_limites(datos)
ca.plantilla_limites(campos_nuevos)

**Qué se pide.** Elige dos casos del banco y escribe su registro esperado completo: los cuatro campos base y tus dos campos nuevos. La celda anterior imprimió una plantilla con tus dos campos. Cópiala en la celda siguiente, en lugar de la lista que trae. Si el ticket no trae un dato, escribe `None`. Para la versión basta el número: «4.3 beta» y «4.3» cuentan igual.

**Por qué.** Un caso límite solo prueba algo si sabes qué debería salir.

**Cómo saber si quedó bien.** La celda dice «Casos límite listos: 2 de 2», sin avisos ni pistas.

* ✏️ Completa tus dos casos límite en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Ejercicio 4: dos casos límite del banco, con su registro esperado
# entrada: déjala en None; el notebook copia el ticket del banco
mis_limites = [
    {"caso_id": ..., "entrada": None, "esperado": {"categoria": ..., "prioridad": ..., "plataforma": ..., "version_app": ...}},
    {"caso_id": ..., "entrada": None, "esperado": {"categoria": ..., "prioridad": ..., "plataforma": ..., "version_app": ...}},
]

* Ejecuta la siguiente celda para revisar tus casos límite.

In [ ]:
# Revisa tus dos casos límite y su registro esperado, campo por campo
limites = ca.armar_limites(datos, mis_limites, campos_nuevos, modo=config.modo)

<a id='7-3'></a>
### 7.3 - Las pruebas otra vez

Ahora corres dos veces los doce tickets y tus dos casos límite. La primera vez, con el esquema de casa sin campos nuevos. La segunda, con tus dos campos. Así ves qué cambió por tus campos y nada más. En vivo, son 28 llamadas.

* Ejecuta la siguiente celda para correr las dos versiones del esquema.

In [ ]:
# Si el Ejercicio 4 está completo, corre los casos sin y con tus dos campos nuevos
sistema_casa = datos["instrucciones"][casa["instruccion"]]
EsquemaSin = ca.armar_ticket(casa["plataforma_obligatoria"], casa["version_obligatoria"])
casos_casa = ca.casos_para_casa(casos, limites, campos_nuevos, datos)
lecturas_sin, lecturas_casa = [], []

if ca.listo_para_casa(campos_nuevos, limites):
    lector_sin = modelo.with_structured_output(EsquemaSin, method="json_schema", include_raw=True)
    lector_casa = modelo.with_structured_output(EsquemaCasa, method="json_schema", include_raw=True)
    print("Sin campos nuevos:")
    lecturas_sin = ca.correr_tickets(lector_sin, config, casos_casa, sistema_casa, EsquemaSin, variante="casa", modelo=modelo)
    print("Con tus dos campos:")
    lecturas_casa = ca.correr_tickets(lector_casa, config, casos_casa, sistema_casa, EsquemaCasa,
                                      variante=ca.variante_de(casa["instruccion"], EsquemaCasa), modelo=modelo)

* Ejecuta la siguiente celda para comparar tu configuración de clase con las dos corridas de casa.

In [ ]:
# Compara tu configuración de clase con las dos corridas de casa, y muestra las salidas inválidas, si hubo
detalle_sin = ca.comparar_campos(lecturas_sin, casos_casa, EsquemaSin, etiqueta="sin campos nuevos")
detalle_casa = ca.comparar_campos(lecturas_casa, casos_casa, EsquemaCasa, etiqueta="con tus dos campos")
if lecturas_casa:
    comparacion = ca.comparar_corridas({"clase: tu configuración": (lecturas, detalle),
                                        "casa: sin campos nuevos": (lecturas_sin, detalle_sin),
                                        "casa: con tus dos campos": (lecturas_casa, detalle_casa)})
    ca.mostrar_campos(detalle_sin, titulo="Los campos de la corrida sin campos nuevos", plegada=True)
    ca.mostrar_campos(detalle_casa, titulo="Los campos de la corrida con tus dos campos")
    for etiqueta, corrida, esquema_corrida in [("sin campos nuevos", lecturas_sin, EsquemaSin),
                                               ("con tus dos campos", lecturas_casa, EsquemaCasa)]:
        invalidas_corrida = [l for l in corrida if not l.valida and not l.error]
        if invalidas_corrida:
            ca.mostrar_lectura(invalidas_corrida[0], config, esquema=esquema_corrida,
                               titulo=f"Una salida inválida de la corrida {etiqueta}: {invalidas_corrida[0].caso_id}")
else:
    print("Todavía no hay corridas de casa para comparar.")

<br>

### Análisis

Mira tres cosas: las salidas válidas, los campos base y tus campos nuevos. Un campo obligatorio que el ticket no trae suele fallar, porque el modelo le pone un valor. Los tokens de entrada suben un poco: el esquema viaja con cada petición. Con una sola corrida, una diferencia de uno o dos campos puede venir de la variación del modelo.

**✏️ Tu análisis (7).** En tres a cinco oraciones, con cifras de la comparación: ¿qué cambió en las salidas válidas y en los campos correctos? ¿Tus casos límite salieron como esperabas? ¿Cambiarías si algún campo es obligatorio?

*Escribe aquí tu análisis.*

<a id='8'></a>
## 8 - Entregable y calificación

*Para casa · 5 minutos*

**Qué entregas.** Este notebook, ejecutado de principio a fin y guardado con sus salidas. Debe tener:
- tus datos de la entrega;
- los cuatro ejercicios resueltos, con sus tablas y sus gráficos;
- tu predicción y tus análisis, en las celdas ✏️;
- la declaración de uso de asistentes de IA y la comprobación de la entrega.

**Cómo se califica.** El laboratorio vale el 2.5 % de la calificación del curso:

| Criterio | Peso | Qué mira el docente |
| :--- | :--- | :--- |
| Funciona según su criterio de éxito | 40 % | El notebook corre y las pruebas corren, sin errores |
| Las pruebas están completas y corridas | 40 % | Las tres etapas y sus pruebas de robustez, con tus registros, tus campos y tus casos límite |
| El análisis está al día | 20 % | Tu predicción, tus análisis y la sección «Tu proyecto» |

**Cómo guardar con las salidas.**
1. Ejecuta todo en orden, hasta la comprobación de la sección 9.
2. Guarda con Ctrl+S (Cmd+S en Mac).
3. Cierra el notebook y vuelve a abrirlo. Si ves las tablas y los gráficos sin ejecutar nada, quedaron guardados.

> **Nota:** no uses «Clear All Outputs» antes de entregar. Esa opción borra las salidas que el docente necesita ver.

**Uso de asistentes de IA.** El curso permite usar asistentes de IA como apoyo, siempre que declares su uso.

* ✏️ Completa la declaración en la celda siguiente y ejecútala.

In [ ]:
# ✏️ Declaración de uso de asistentes de IA
declaracion_ia = {
    "usaste_asistentes": None,   # True o False, sin comillas
    "cuales_y_para_que": "...",  # por ejemplo: "Claude, para entender un mensaje de error de Pydantic"
}

<a id='9'></a>
## 9 - Comprobación de la entrega

*Para casa · 5 minutos*

La celda siguiente revisa que la entrega esté completa. Muestra «listo», «revisa» o «falta» en cada parte. «Revisa» quiere decir que la parte está, pero tiene algo por corregir. Orienta, no califica. Como no puede leer las celdas de texto, tu predicción y tus análisis los revisas tú.

* Ejecuta la siguiente celda para comprobar tu entrega.

In [ ]:
# Revisa que la entrega esté completa: muestra «listo», «revisa» o «falta» en cada parte
entrega_completa = ca.comprobar_entrega_3A(
    datos_entrega=datos_entrega, config=config, instruccion=INSTRUCCION,
    plataforma_obligatoria=PLATAFORMA_OBLIGATORIA, version_obligatoria=VERSION_OBLIGATORIA,
    casos=casos, lecturas=lecturas, vacio=vacio, cortada=cortada, detalle=detalle, revision=revision,
    campos_nuevos=campos_nuevos, limites=limites, lecturas_casa=lecturas_casa, declaracion_ia=declaracion_ia,
)

<a id='10'></a>
## 10 - Tu proyecto: la capa 2

*Para casa · 10 minutos*

**Qué capa avanza.** Este laboratorio avanza la capa 2 de tu proyecto: las instrucciones. Su resultado va en la sección 2 de la ficha del agente.

**Cómo reutilizar este notebook.** Cambia el esquema `Ticket` por el de la tarea central de tu agente. Cambia los tickets por los diez casos de la capa 1, que tu grupo escribió en la ficha del agente. Cada uno necesita su registro esperado. Corre las tres etapas y lleva la tabla de resultados a la ficha. Por ejemplo:
- si tu agente **clasifica**, el esquema trae la categoría con sus valores permitidos;
- si **extrae datos**, cada dato es un campo, y los que pueden faltar admiten vacío;
- si **resume**, el esquema separa los datos que no pueden faltar del texto libre;
- si **responde preguntas**, el esquema trae la respuesta y la fuente.

Prevé un valor para las entradas que no aplican, como una categoría «fuera de alcance». Si no, el modelo forzará una de las otras.

**Qué llevas a la ficha.** El esquema, con cada campo y si es obligatorio. También los registros esperados y los resultados de la validación: el porcentaje de salidas válidas y el de campos correctos. Usa datos sintéticos o anonimizados. Ningún campo del esquema pide datos personales: ni nombres, ni correos, ni teléfonos.

**Qué sigue.** El Laboratorio 3B compara variantes del prompt de sistema. La capa 3, en la sesión 4, pone tu llamada dentro del ciclo de un agente.

**✏️ Tu proyecto.** En dos o tres oraciones: ¿qué campos tendrá el esquema de tu agente? ¿Cuál puede quedar vacío, y por qué?

*Escribe aquí tu respuesta.*

<a id='conclusion'></a>
## Conclusión

En este laboratorio pasaste un ticket de texto libre a un registro que cumple un esquema. Leíste el esquema como una clase de Pydantic y decidiste qué campos son obligatorios. Probaste tres etapas: la respuesta cruda, el esquema y los campos. Viste que la forma se puede garantizar, pero el contenido se prueba. También viste que un campo obligatorio obliga al modelo a poner un valor, aunque el ticket no lo traiga.

El Laboratorio 3B compara variantes del prompt sobre los mismos casos, con sus aciertos y su costo.

<a id='referencias'></a>
## Referencias

**Del bloque 3.1**
- Anthropic. *Prompt engineering overview* (en español). https://platform.claude.com/docs/es/build-with-claude/prompt-engineering/overview
- Anthropic. *Structured outputs*. https://platform.claude.com/docs/en/build-with-claude/structured-outputs
- Google. *Structured output*. Gemini API. https://aistudio.google.com/docs/structured-output?generate-content=true
- OpenAI (2024, 6 de agosto). *Introducing Structured Outputs in the API*. [Original](https://openai.com/index/introducing-structured-outputs-in-the-api/) · [en español](https://openai.com/es-ES/index/introducing-structured-outputs-in-the-api/).
- Huyen, C. (2024). *AI Engineering*, capítulo 5. O'Reilly.

**De la capa de evaluación del curso (Anexo J)**
- Grace, M., Hadfield, J., Olivares, R. y De Jonghe, J. (2026, 9 de enero). *Demystifying evals for AI agents*. Anthropic. https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents

**Documentación de las herramientas**, consultada el 24 de noviembre de 2026
- LangChain. *Structured output*. https://docs.langchain.com/oss/python/langchain/structured-output
- LangChain. *Unit testing*: los modelos simulados. https://docs.langchain.com/oss/python/langchain/test/unit-testing
- Pydantic. *Documentation*. https://docs.pydantic.dev/
- Anthropic. *Claude Haiku 5.5: migration guide*. https://platform.claude.com/docs/en/models/haiku-5-5/migration-guide
- Visual Studio Code. *Jupyter Notebooks in VS Code*. https://code.visualstudio.com/docs/datascience/jupyter-notebooks